In [1]:
import requests
import time

In [2]:
# retrieve publication data from semantic scholar database using api
# api_key might expire/break; make sure api_key is working properly

def get_publication_data(api_key, query, target, limit, offset):
    endpoint = 'https://api.semanticscholar.org/graph/v1/paper/search'
    headers = {'x-api-key': api_key}
    params = {
        'query': query,
        'limit': limit,
        'offset': offset,

        'fields': ','.join([
            'paperId',
            'externalIds',  # this dictionary object contains 'DOI' key
            'title',
            'citationCount',
            'influentialCitationCount',
            'isOpenAccess',
            'openAccessPdf'  # this dictionary object contains 'url' key
        ])
    }

    delay = 10
    last_offset = 0
    next_offset = 0
    open_access_publications = []
    
    while len(open_access_publications) <= target:
        response = requests.get(
            endpoint,
            params=params,
            headers=headers
        )
    
        if response.status_code == 200:
            data = response.json()
            publications = data['data']
            last_offset = data['offset']
            next_offset = data['next']
            
            for pub in publications:
                if pub['isOpenAccess']:
                    open_access_publications.append(pub)
            
            params['offset'] += params['limit']
            print(f'Collected data: {len(open_access_publications)}')
        
        elif response.status_code == 429:
            print(f'Too many requests! Waiting for {delay}s...')
            time.sleep(10)
        
        else:
            print(f'Error: {response.status_code}! Waiting for {delay}s...')
            time.sleep(10)

    if len(open_access_publications) >= target:
        print('\nTarget reached successfully.')
        print(f'Total collected data: {len(open_access_publications)}')
        print(f'Last offset used: {last_offset}')
        print(f'Next offset to set: {next_offset}')

    return open_access_publications


In [3]:
# download pdf from its source 
# downloaded pdfs may still corrupted (unable to open)

def get_pdf(publication_list):
    success_list = []
    failed_list = []
    null_url_list = []
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    for pub in publication_list:
        if pub['openAccessPdf'] is not None:
            url = pub['openAccessPdf']['url']
            
            try:
                response = requests.get(url, headers=headers, timeout=20)
    
                if response.status_code == 200:  # status code 200 is successful response
                    pdf_content = response.content
    
                    with open(f'pdfs/paper_id_{pub['paperId']}.pdf', 'wb') as pdf_file:
                        pdf_file.write(pdf_content)
    
                    success_list.append(pub)
    
                else:
                    failed_list.append(pub)
                    print(f'{response.status_code}:{response.reason}')

            except requests.exceptions.RequestException as e:
                failed_list.append(pub)
                print(f'Request failed: {e}')

            time.sleep(2)
            
        else:
            print(f'No URL found!')
            null_url_list.append(pub)

    print(f'\nSuccessful download: {len(success_list)}')
    print(f'Failed download: {len(failed_list)}')
    print(f'No URL found: {len(null_url_list)}')
    
    return success_list, failed_list, null_url_list
    

In [4]:
# set the api_key

def main():
    api_key = ''
    query = 'cellulose materials'  # search keyword
    target = 200  # number of open access publication data to retrieve
    limit = 50  # number of publication data retrieves per api request
    offset = 0  # starting index to retrieve publication data

    print('\nCollecting publication data...\n')
    publication_list = get_publication_data(
        api_key=api_key,
        query=query,
        target=target,
        limit=limit,
        offset=offset
    )

    print('\nDownloading PDFs...\n')
    success_list, failed_list, null_url_list = get_pdf(publication_list)

    print(f'\nORIGINAL LIST: {publication_list}')
    print(f'\nFAILED LIST: {failed_list}')


if __name__ == '__main__':
    main()



Collected data: 17
Collected data: 40
Collected data: 54
Collected data: 69
Collected data: 92
Collected data: 116
Collected data: 135
Collected data: 152
Collected data: 169
Collected data: 180
Too many requests! Waiting for 10s...
Collected data: 198
Collected data: 212

Target reached successfully.
Total collected data: 212
Last offset used: 550
Next offset to set: 600


403:Forbidden
403:Forbidden
403:Forbidden
403:Forbidden
403:Forbidden
202:Accepted
403:Forbidden
403:Forbidden
403:Forbidden
403:Forbidden
403:Forbidden
403:Forbidden
403:Forbidden
403:Forbidden
403:Forbidden
403:Forbidden
405:Not Allowed
403:Forbidden
403:Forbidden
403:Forbidden
Request failed: HTTPSConnectionPool(host='cronfa.swan.ac.uk', port=443): Read timed out. (read timeout=20)
403:Forbidden
403:Forbidden
403:Forbidden
403:Forbidden
403:Forbidden
403:Forbidden
403:Forbidden
403:Forbidden
403:Forbidden
403:Forbidden
403:Forbidden
403:Forbidden
Request failed: HTTPSConnectionPool(host='riunet.upv.es', port=44